# K近邻（KNN）算法示例：预测设备故障

本 notebook 使用 `predictive_maintenance.csv` 演示 K 近邻（K-Nearest Neighbors, KNN）分类的基本流程：

1. 读取工业设备传感器数据；
2. 选择特征与目标变量；
3. 拆分训练集和测试集；
4. 对数值特征做标准化，对类别特征做 One-Hot 编码；
5. 训练 KNN 分类器；
6. 评估混淆矩阵、Precision、Recall、F1；
7. 调整 K 值，并比较普通投票与距离加权投票。

## 数据集说明

目标变量是：

- `Machine failure`：设备是否故障，`1` 表示故障，`0` 表示正常。

候选特征包括：

- `Type`：设备类型；
- `Air temperature`：空气温度；
- `Process temperature`：过程温度；
- `Rotational speed`：转速；
- `Torque`：扭矩；
- `Tool wear`：刀具磨损。

`TWF`、`HDF`、`PWF`、`OSF`、`RNF` 是故障原因或故障模式指示变量，与目标变量关系过强，直接作为特征可能造成标签泄露，因此本示例不把它们作为输入特征。

## 1. KNN 的核心思想

KNN 是一种“基于实例”的算法。预测一个新样本时，它不是先显式学习一组模型参数，而是：

1. 计算新样本到训练集中每个样本的距离；
2. 找出距离最近的 `K` 个样本；
3. 分类任务中采用多数投票：哪个类别占比最高，就预测为哪个类别。

例如，`K=5` 时，距离最近的 5 个训练样本里有 3 个故障、2 个正常，则新样本被预测为故障。

### 为什么必须标准化？

KNN 依赖距离。若某个特征数值范围很大，它会主导距离计算。例如转速是千级，而温度是百级。如果不做标准化，KNN 很容易被量纲大的特征带偏。

### K 值如何选择？

- `K` 太小：模型容易受噪声影响，可能过拟合；
- `K` 太大：决策边界变得平滑，可能欠拟合；
- 二分类中通常优先考虑奇数 `K`，减少平票；
- 实际选择应结合验证集或交叉验证，并关注业务更看重的指标。

本数据集故障样本只占约 3.4%，所以不能只看准确率，还要重点关注故障类别的 Recall、Precision 和 F1。

## 2. 导入库

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

## 3. 读取数据

In [ ]:
csv_path = Path.cwd() / "predictive_maintenance.csv"

if not csv_path.exists():
    csv_path = Path("Python Guidance/day31-45/predictive_maintenance.csv")

df = pd.read_csv(csv_path)
df.head()

## 4. 定义特征和目标变量

这里用传感器测量值和设备类型预测 `Machine failure`。

注意：不把 `TWF`、`HDF`、`PWF`、`OSF`、`RNF` 作为输入特征，因为它们本质上是故障原因/故障模式，会带来标签泄露。

In [ ]:
target = "Machine failure"
failure_mode_columns = ["TWF", "HDF", "PWF", "OSF", "RNF"]

numeric_features = [
    "Air temperature",
    "Process temperature",
    "Rotational speed",
    "Torque",
    "Tool wear",
]
categorical_features = ["Type"]

X = df[numeric_features + categorical_features]
y = df[target]

## 5. 初步检查数据

In [ ]:
print("数据形状:", df.shape)
print("缺失值数量:", int(df.isna().sum().sum()))
print("\n目标变量分布:")
print(y.value_counts().rename(index={0: "正常", 1: "故障"}))
print(f"\n故障比例: {y.mean():.2%}")

In [ ]:
df[numeric_features].describe().T

可以看到，正常样本远多于故障样本。也就是说，即使模型把所有样本都预测成“正常”，准确率仍会很高。因此在这个任务中，故障类别的 Recall 和 F1 更重要。

- Recall（召回率）：真实故障中，有多少被模型找出来；
- Precision（精确率）：模型预测为故障的样本中，有多少真的是故障；
- F1：Precision 和 Recall 的调和平均，适合在类别不平衡时综合评估。

## 6. 拆分训练集和测试集

`stratify=y` 会让训练集和测试集中的故障比例接近原始数据，避免随机拆分时故障样本分布过于集中。

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("训练集:", X_train.shape)
print("测试集:", X_test.shape)
print("训练集故障比例:", f"{y_train.mean():.2%}")
print("测试集故障比例:", f"{y_test.mean():.2%}")

## 7. 建立 KNN Pipeline

这个 Pipeline 包含两步：

1. 预处理：数值特征标准化，类别特征 `Type` 做 One-Hot 编码；
2. 模型：`KNeighborsClassifier`，先使用 `K=5`。

把预处理和模型放进 `Pipeline`，可以保证标准化统计量只来自训练集，再应用到测试集。

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

knn = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=5, weights="uniform")),
    ]
)

knn.fit(X_train, y_train)

## 8. 在测试集上评估

In [ ]:
y_pred = knn.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["正常", "故障"], digits=3))

In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["正常", "故障"],
    cmap="Blues",
    values_format="d",
)
plt.title("KNN 混淆矩阵（K=5）")
plt.show()

混淆矩阵中：

- 左上：真实正常，预测正常；
- 右上：真实正常，预测故障，属于误报；
- 左下：真实故障，预测正常，属于漏报；
- 右下：真实故障，预测故障，属于成功识别故障。

工业预测性维护中，漏报通常代价较高，因为设备已经故障但模型没有发现；误报则会带来额外检查成本。

## 9. 调整 K 值

下面比较多个 K 值下的表现。由于数据类别不平衡，这里重点观察故障类别的 F1 和 Recall。

In [ ]:
k_values = [1, 3, 5, 7, 9, 15, 21, 31]
rows = []

for k in k_values:
    model = Pipeline(
        steps=[
            ("prep", preprocessor),
            ("model", KNeighborsClassifier(n_neighbors=k, weights="uniform")),
        ]
    )
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    rows.append(
        {
            "k": k,
            "accuracy": model.score(X_test, y_test),
            "precision": precision_score(y_test, pred, zero_division=0),
            "recall": recall_score(y_test, pred),
            "f1": f1_score(y_test, pred, zero_division=0),
        }
    )

k_results = pd.DataFrame(rows)
k_results

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(k_results["k"], k_results["precision"], marker="o", label="Precision")
plt.plot(k_results["k"], k_results["recall"], marker="o", label="Recall")
plt.plot(k_results["k"], k_results["f1"], marker="o", label="F1")
plt.xlabel("K")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title("不同 K 值下的测试集表现")
plt.show()

In [ ]:
best_k = int(k_results.sort_values("f1", ascending=False).iloc[0]["k"])
best_f1 = k_results["f1"].max()

print(f"按 F1 选择的 K: {best_k}")
print(f"对应 F1: {best_f1:.3f}")

## 10. 距离加权 KNN

普通 KNN 中，最近的 1 个邻居和第 K 个邻居拥有相同投票权。`weights="distance"` 则让距离越近的邻居权重越大，通常会让决策边界更关注局部样本。

In [ ]:
weighted_knn = Pipeline(
    steps=[
        ("prep", preprocessor),
        ("model", KNeighborsClassifier(n_neighbors=best_k, weights="distance")),
    ]
)

weighted_knn.fit(X_train, y_train)
y_pred_weighted = weighted_knn.predict(X_test)

print(classification_report(y_test, y_pred_weighted, target_names=["正常", "故障"], digits=3))
print(f"f1: {f1_score(y_test, y_pred_weighted, zero_division=0)}")


## 11. 调整概率阈值：减少漏报还是减少误报

KNN 的 `predict` 默认等价于使用 `0.5` 作为阈值：

- 故障概率 `>= 0.5`：预测为故障；
- 故障概率 `< 0.5`：预测为正常。

但在工业设备维护中，漏掉一次真实故障的代价，可能远高于一次误报。因此可以主动调整阈值：

- **降低阈值**：更多样本会被判定为故障，通常能减少漏报（FN 下降），但会增加误报（FP 上升）；
- **提高阈值**：更少样本会被判定为故障，通常能减少误报（FP 下降），但会增加漏报（FN 上升）。

下面用距离加权 KNN 的 `predict_proba` 输出，比较不同阈值下的业务权衡。

In [ ]:
import numpy as np

failure_proba = weighted_knn.predict_proba(X_test)[:, 1]
thresholds = np.round(np.arange(0.10, 0.91, 0.05), 2)

rows = []
for threshold in thresholds:
    y_pred_threshold = (failure_proba >= threshold).astype(int)
    y_true_array = y_test.to_numpy()

    tp = int(((y_pred_threshold == 1) & (y_true_array == 1)).sum())
    tn = int(((y_pred_threshold == 0) & (y_true_array == 0)).sum())
    fp = int(((y_pred_threshold == 1) & (y_true_array == 0)).sum())
    fn = int(((y_pred_threshold == 0) & (y_true_array == 1)).sum())

    rows.append(
        {
            "threshold": threshold,
            "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_test, y_pred_threshold),
            "f1": f1_score(y_test, y_pred_threshold, zero_division=0),
            "false_positives": fp,
            "false_negatives": fn,
            "flagged_as_failure": tp + fp,
        }
    )

threshold_results = pd.DataFrame(rows)
threshold_results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(
    threshold_results["threshold"],
    threshold_results["precision"],
    marker="o",
    label="Precision",
)
axes[0].plot(
    threshold_results["threshold"],
    threshold_results["recall"],
    marker="o",
    label="Recall",
)
axes[0].plot(
    threshold_results["threshold"],
    threshold_results["f1"],
    marker="o",
    label="F1",
)
axes[0].axvline(0.5, color="gray", linestyle="--", label="默认阈值 0.5")
axes[0].set_xlabel("故障概率阈值")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.05)
axes[0].grid(True, alpha=0.3)
axes[0].legend()
axes[0].set_title("阈值与分类指标")

axes[1].plot(
    threshold_results["threshold"],
    threshold_results["false_positives"],
    marker="o",
    label="误报 FP",
)
axes[1].plot(
    threshold_results["threshold"],
    threshold_results["false_negatives"],
    marker="o",
    label="漏报 FN",
)
axes[1].axvline(0.5, color="gray", linestyle="--", label="默认阈值 0.5")
axes[1].set_xlabel("故障概率阈值")
axes[1].set_ylabel("数量")
axes[1].grid(True, alpha=0.3)
axes[1].legend()
axes[1].set_title("阈值与误报/漏报数量")

plt.tight_layout()
plt.show()

## 12. 预测新样本

下面构造一条新记录，演示如何使用训练好的 KNN 模型进行预测。

`predict_proba` 返回的是模型认为样本属于各个类别的概率。由于这是 KNN 的投票/加权投票结果，可以把它理解为一种“邻居投票得分”。

In [ ]:
new_sample = pd.DataFrame(
    [
        {
            "Type": "M",
            "Air temperature": 299.0,
            "Process temperature": 311.0,
            "Rotational speed": 1650,
            "Torque": 55.0,
            "Tool wear": 180,
        }
    ]
)

failure_probability = weighted_knn.predict_proba(new_sample)[0, 1]
default_prediction = weighted_knn.predict(new_sample)[0]

business_threshold = 0.30
business_prediction = int(failure_probability >= business_threshold)

print(f"故障概率得分: {failure_probability:.3f}")
print("默认阈值 0.5:", "故障" if default_prediction == 1 else "正常")
print(f"业务阈值 {business_threshold:.2f}:", "故障" if business_prediction == 1 else "正常")
new_sample

## 13. 总结

本示例展示了 KNN 在设备故障预测中的完整流程：

- KNN 通过“最近邻投票”完成分类；
- 数值特征标准化对 KNN 非常重要；
- `Type` 这类类别特征需要先编码再输入模型；
- 故障样本比例很低时，准确率会掩盖模型问题，应重点看故障类别的 Recall、Precision 和 F1；
- K 值会直接影响模型的偏差和方差，应通过验证结果选择；
- `weights="distance"` 是一种常见的加权投票方式，但并不保证在所有数据集上都优于普通投票；
- `predict_proba` 配合自定义阈值，可以在“减少漏报”和“减少误报”之间做业务权衡。

## 可以进一步尝试

1. 用交叉验证选择 K 和阈值，而不是只用一次训练/测试划分；
2. 尝试对少数类过采样或类别加权方法；
3. 比较 KNN 与逻辑回归、随机森林、梯度提升树等模型；
4. 使用 PCA 或特征选择减少噪声特征，观察 KNN 是否更稳定；
5. 根据实际业务成本建立成本矩阵，选择总成本最低的阈值。